# 상용화 테스트

In [13]:
import os
import torch
import torch.nn as nn
import librosa
import numpy as np
import noisereduce as nr
from transformers import AutoModel, AutoFeatureExtractor

In [14]:
# ---------------------------------------------------------
# 1. 모델 구조 정의 (학습할 때와 동일해야 가중치가 로드됨)
# ---------------------------------------------------------
class MultiTaskAudioRegression(nn.Module):
    def __init__(self, model_name="facebook/hubert-base-ls960", dropout_prob=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        
        hidden_size = self.backbone.config.hidden_size
        
        # Heads
        self.artic_head = nn.Sequential(
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )
        
        self.prosody_head = nn.Sequential(
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )
        
    def forward(self, input_values, attention_mask=None):
        outputs = self.backbone(input_values=input_values, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        pooled_output = torch.mean(last_hidden_state, dim=1)
        
        artic_pred = self.artic_head(pooled_output)
        prosody_pred = self.prosody_head(pooled_output)
        
        return artic_pred, prosody_pred

In [15]:
# ---------------------------------------------------------
# 2. 추론 파이프라인 클래스 (서버 백엔드용)
# ---------------------------------------------------------
class SpeechEvaluator:
    def __init__(self, model_weights_path):
        """
        서버 시작 시 1번만 실행되는 초기화 함수 (모델 및 프로세서 로드)
        """
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"🚀 Initialize Pipeline on {self.device}...")
        
        self.sr = 16000
        self.max_length = int(self.sr * 8.0) # 학습 시 max_seconds=8.0
        self.padding_samples = int(self.sr * 0.25) # VAD 패딩 250ms
        
        # 1. HuBERT 프로세서 로드
        self.processor = AutoFeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
        
        # 2. Silero VAD 모델 로드
        print("Loading Silero VAD...")
        self.model_vad, self.vad_utils = torch.hub.load(
            repo_or_dir='snakers4/silero-vad',
            model='silero_vad',
            force_reload=False
        )
        self.get_speech_timestamps = self.vad_utils[0]
        self.model_vad.to(self.device)
        
        # 3. 메인 평가 모델 로드 및 가중치 덮어쓰기
        print("Loading Evaluator Model...")
        self.model = MultiTaskAudioRegression(model_name="facebook/hubert-base-ls960").to(self.device)
        self.model.load_state_dict(torch.load(model_weights_path, map_location=self.device))
        self.model.eval() # 평가 모드로 전환 (Dropout 등 비활성화)
        
        print("✅ Pipeline Ready!")

    def preprocess_audio(self, audio_path):
        """
        음성 전처리: Peak Norm -> VAD -> Denoise
        """
        # 1. 오디오 로드
        wav_orig, _ = librosa.load(audio_path, sr=self.sr, mono=True, dtype=np.float32)
        
        # 2. Peak 정규화
        wav_peak_norm = librosa.util.normalize(wav_orig).astype(np.float32)
        
        # 3. VAD 적용
        wav_tensor = torch.from_numpy(wav_peak_norm).to(self.device)
        timestamps = self.get_speech_timestamps(wav_tensor, self.model_vad, sampling_rate=self.sr, threshold=0.5)
        
        if not timestamps:
            wav_vad = wav_peak_norm # VAD 실패 시 원본(정규화) 그대로 사용
        else:
            start = max(0, timestamps[0]['start'] - self.padding_samples)
            end = min(len(wav_peak_norm), timestamps[-1]['end'] + self.padding_samples)
            wav_vad = wav_peak_norm[start:end]
            
        # 4. Denoising (학습 때와 동일하게 0.4)
        #wav_denoised = nr.reduce_noise(y=wav_vad, sr=self.sr, prop_decrease=0.4)
        
        #return wav_denoised
        return wav_vad

    def predict(self, audio_path):
        """
        전처리 및 모델 추론을 거쳐 최종 점수를 반환
        """
        # 1. 전처리 실행
        try:
            clean_audio = self.preprocess_audio(audio_path)
        except Exception as e:
            return {"error": f"Preprocessing failed: {str(e)}"}
            
        # 2. HuBERT Processor 적용 (학습 Dataset의 __getitem__ 과 동일)
        inputs = self.processor(
            clean_audio, 
            sampling_rate=self.sr, 
            return_tensors="pt", 
            padding='max_length', 
            max_length=self.max_length,
            truncation=True
        )
        
        input_values = inputs.input_values.to(self.device)
        
        # 3. 모델 추론
        with torch.no_grad():
            artic_pred, prosody_pred = self.model(input_values)
            
        # 4. 결과값 후처리 (소수점 둘째 자리까지 반올림, float 변환)
        artic_score = round(artic_pred.item(), 2)
        prosody_score = round(prosody_pred.item(), 2)
        
        # 예외 처리: 점수가 1.0~5.0 범위를 벗어나지 않도록 클리핑 (옵션)
        artic_score = max(1.0, min(5.0, artic_score))
        prosody_score = max(1.0, min(5.0, prosody_score))
        
        return {
            "articulation_score": artic_score,
            "prosody_score": prosody_score
        }

In [16]:
# 1. 서버 부팅 시 객체 생성 (무거운 모델 로딩은 여기서 끝남)
MODEL_PATH = r"C:\Users\ohje\Desktop\오제석\best_model\hubert_721_740.pt"
evaluator = SpeechEvaluator(model_weights_path=MODEL_PATH)

# 2. 유저가 웹/앱에서 음성 파일을 업로드했을 때
test_audio_file = r"C:\Users\ohje\Desktop\오제석\Sample\01.원천데이터\en\PRN\NA\NA\NA-NA-F-13-ko-220812-100000_133_1_19-en.wav"

if os.path.exists(test_audio_file):
    print("\n⏳ 평가를 진행 중입니다...")
    
    # 3. 점수 예측 요청
    result = evaluator.predict(test_audio_file)
    
    print("\n📊 [평가 결과]")
    print(f"발음 (Articulation): {result['articulation_score']} / 5.0")
    print(f"운율 (Prosody):      {result['prosody_score']} / 5.0")
else:
    print(f"테스트용 음성 파일이 없습니다: {test_audio_file}")

🚀 Initialize Pipeline on cuda...
Loading Silero VAD...
Loading Evaluator Model...


Using cache found in C:\Users\ohje/.cache\torch\hub\snakers4_silero-vad_master


✅ Pipeline Ready!

⏳ 평가를 진행 중입니다...

📊 [평가 결과]
발음 (Articulation): 4.18 / 5.0
운율 (Prosody):      4.22 / 5.0
